In [ ]:
from backend.src.services.retrieval.vector_service import VectorService
from backend.src.domain.schemas.config import VectorStoreConfig,EmbedderConfig, ChunkerConfig
from backend.src.services.processing.chunker_service import ChunkerService
from backend.src.services.processing.embedder_service import EmbedderService
from backend.src.services.processing.loader_service import (LoaderService, to_bytes)
from backend.src.domain.schemas.doc import (Doc, MDNFile, PDFMetadata, TextMetadata)
from backend.src.services.processing.hi_chunk import HiChunk, HiChunkStructurer

import asyncio
from rich.pretty import pprint as pp

vs = VectorService()
ls = LoaderService()
cs = ChunkerService()
es = EmbedderService()
vc = VectorStoreConfig()
ec = EmbedderConfig()
cc = ChunkerConfig()
hcs = HiChunkStructurer(config=cc)
md: str = "/home/roccoluxe/Documents/docs/04-languages-types/typescript/02-type-system/Basic Types.md"

In [ ]:
doc: list[Doc] = ls._parse_files([md])
pp(doc)

In [ ]:
from typing import TYPE_CHECKING, Any

import litellm
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_litellm import ChatLiteLLM
from loguru import logger

from backend.src.api.schemas.api_schemas import UIMessage
from backend.src.domain.schemas.config import (
    LLMConfig,
    ScoredRetrieval,
    UserPreferences,
)
from backend.src.domain.schemas.doc import Chunk, Doc
from backend.src.services.conversation.utils.llm_utils import (
    build_model_string,
    filter_none_values,
    format_sources_section,
    langchain_chat_history_to_str,
    ui_messages_to_lc_messages,
)
from backend.src.services.conversation.utils.prompts import FOSRA_SYSTEM_PROMPT, GRAPH_SEARCH_PROMPT_TEMPLATE, \
    GRAPH_QUERY_GEN_PROMPT_TEMPLATE, QUERY_REFORM_PROMPT, SPLIT_SUBQUERIES_PROMPT_TEMPLATE

if TYPE_CHECKING:
    pass

litellm.drop_params = True


PROVIDER_TO_LITELLM_MAP: dict[str, str] = {
    "OPENAI": "openai",
    "ANTHROPIC": "anthropic",
    "COHERE": "cohere",
    "GROQ": "groq",
    "TOGETHER": "together_ai",
    "MISTRAL": "mistral",
    "REPLICATE": "replicate",
    "HUGGINGFACE": "huggingface",
    "BEDROCK": "bedrock",
    "VERTEX_AI": "vertex_ai",
    "PALM": "palm",
    "OPENROUTER": "openrouter",
}


from backend.src.services.conversation.utils.prompts import (
    COVERAGE_CHECK_PROMPT,
    DOC_TOPIC_GEN_PROMPT,
    FOSRA_CITATION_INSTRUCTIONS,
    GRAPH_QUERY_GEN_PROMPT,
    GRAPH_SEARCH_PROMPT,
    SPLIT_SUBQUERIES_PROMPT,
    SPLIT_SUBQUERIES_PROMPT_TEMPLATE,
    COVERAGE_CHECK_PROMPT_TEMPLATE,
)
MOCK_TOPICS = [
    # TanStack Query / general data fetching docs
    "data_fetching",
    "cache_invalidation",
    "cache_mutation",
    "configuration",
    "pagination",
    "error_handling",
    "optimistic_updates",
    "prefetching",
    "query_filters",
    "subscriptions",
    "devtools",
    "ssr_hydration",
    # Source code structural
    "authentication",
    "data_transformation",
    "file_io",
    "api_client",
    "middleware",
    "database_access",
    "validation",
    "event_handling",
    "state_management",
    "routing",
    # Meta / navigation — always present
    "navigation",
]

c: LLMConfig = LLMConfig(
    config_id=0,
    config_name="the config name",
    provider="openrouter",
    model="openai/gpt-oss-20b:nitro",
    api_key="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",
    api_base="https://openrouter.ai/api/v1",
)

ms: str = build_model_string(
    provider=c.provider,
    model_name=c.model,
    custom_provider=c.custom_provider,
)

kw: dict[str, Any] = {
    "model": ms,
    "api_key": c.api_key,
    "streaming": True,
}

if c.api_base:
    kw["api_base"] = c.api_base

if c.litellm_params:
    kw.update(c.litellm_params)


llm: ChatLiteLLM = ChatLiteLLM(**filter_none_values(kw))



In [ ]:
#DOC TOPIC GENERATION
prompt = DOC_TOPIC_GEN_PROMPT.format(
    existing_topics=MOCK_TOPICS, chunk_text=doc
)
res = llm.astream(input=prompt)

async for chunk in res:
    print(chunk.content, end="", flush=True)

In [ ]:
#GRAPH SEMANITC SEARCH QUERY GENERATION

prompt = GRAPH_SEARCH_PROMPT_TEMPLATE.replace("{query}", "what is the token_spiller class doing?")

res = llm.astream(input=prompt)

async for chunk in res:
    print(chunk.content, end="", flush=True)


In [ ]:
#DOC CYPYER QUERY GENERATION
# prompt = GRAPH_QUERY_GEN_PROMPT.format(
#     user_query="what does the token_separator class do?"
# )

prompt = GRAPH_QUERY_GEN_PROMPT_TEMPLATE.replace("{query}", "what does the token separator function call?")

res = llm.astream(input=prompt)

async for chunk in res:
    print(chunk.content, end="", flush=True)

In [ ]:
#QUERY REFORMATION

prompt = QUERY_REFORM_PROMPT.replace("{user_query}", "what does the token separator function call?")

res = llm.astream(input=prompt)

async for chunk in res:
    print(chunk.content, end="", flush=True)

In [ ]:
prompt = SPLIT_SUBQUERIES_PROMPT_TEMPLATE.replace("{user_query}", "what does the token separator function call?")

res = llm.astream(input=prompt)

split_prompts = []
async for chunk in res:
    split_prompts.append(chunk.content)
    print(chunk.content, end="", flush=True)

In [55]:
search = await vs.search(config=vc, embed_config=ec,query="what is the never type")
pp(search)

2026-03-13 19:26:36.915 | DEBUG    | backend.src.services.processing.embedder_service:embed_query:190 - Embedding query using FASTEMBED
2026-03-13 19:26:36.915 | DEBUG    | backend.src.services.processing.embedder_service:_get_embedders:57 - Initializing FASTEMBED embedder with model
2026-03-13 19:26:37.664 | INFO     | backend.src.services.processing.embedder_service:_get_embedders:95 - Initialized FASTEMBED embedders: Dense: True | Sparse: True Late: True
2026-03-13 19:26:37.746 | INFO     | backend.src.services.processing.embedder_service:embed_query:224 - Queries Embedded: Dense: True | Sparse: True Late: True
2026-03-13 19:26:37.773 | DEBUG    | backend.src.services.retrieval.vector_service:search:267 - ENTERED RETRIEVAL LATE


DEBUGPRINT[97]: vector_service.py:288 (before results = store.query_points()
DEBUGPRINT[101]: vector_service.py:301 (before return results)


QueryResponse(
│   points=[
│   │   ScoredPoint(
│   │   │   id='1368b9c2-955a-4210-ac70-fb75a63a2400',
│   │   │   version=1,
│   │   │   score=26.720034,
│   │   │   payload={
│   │   │   │   'text': '\n## Never\n\nThe `never` type represents the type of values that never occur.\nFor instance, `never` is the return type for a function expression or an arrow function expression that always throws an exception or one that never returns.\nVariables also acquire the type `never` when narrowed by any type guards that can never be true.\n',
│   │   │   │   'source_id': None,
│   │   │   │   'token_count': 72,
│   │   │   │   'start_char': 9415,
│   │   │   │   'end_char': 9746,
│   │   │   │   'parent_text': '\n## Never\n\nThe `never` type represents the type of values that never occur.\nFor instance, `never` is the return type for a function expression or an arrow function expression that always throws an exception or one that never returns.\nVariables also acquire the type `never` when narrowed by any type guards that can never be true.\n',
│   │   │   │   'parent_token_count': 72,
│   │   │   │   'parent_start_char': 9415,
│   │   │   │   'parent_end_char': 9746,
│   │   │   │   'parent_level': 2,
│   │   │   │   'grandparent_text': '\n## Never\n\nThe `never` type represents the type of values that never occur.\nFor instance, `never` is the return type for a function expression or an arrow function expression that always throws an exception or one that never returns.\nVariables also acquire the type `never` when narrowed by any type guards that can never be true.\n\nThe `never` type is a subtype of, and assignable to, every type; however, _no_ type is a subtype of, or assignable to, `never` (except `never` itself).\nEven `any` isn\'t assignable to `never`.\n\nSome examples of functions returning `never`:\n\n```ts twoslash\n// Function returning never must not have a reachable end point\nfunction error(message: string): never {\n  throw new Error(message);\n}\n\n// Inferred return type is never\nfunction fail() {\n  return error("Something failed");\n}\n\n// Function returning never must not have a reachable end point\nfunction infiniteLoop(): never {\n  while (true) {}\n}\n```\n',
│   │   │   │   'grandparent_token_count': 238,
│   │   │   │   'grandparent_start_char': 9415,
│   │   │   │   'grandparent_level': 1,
│   │   │   │   'parent_id': 'None:9415:9746',
│   │   │   │   'grandparent_id': 'None:9415:10349'
│   │   │   },
│   │   │   vector=None,
│   │   │   shard_key=None,
│   │   │   order_value=None
│   │   ),
│   │   ScoredPoint(
│   │   │   id='b3a6b7b3-81cf-4641-8524-7aa24771ae5e',
│   │   │   version=1,
│   │   │   score=19.621586,
│   │   │   payload={
│   │   │   │   'text': '\n\n## About `Number`, `String`, `Boolean`, `Symbol` and `Object`\n\nIt can be tempting to think that the types `Number`, `String`, `Boolean`, `Symbol`, or `Object` are the same as the lowercase versions recommended above.\nThese types do not refer to the language primitives however, and almost never should be used as a type.\n\n',
│   │   │   │   'source_id': None,
│   │   │   │   'token_count': 82,
│   │   │   │   'start_char': 12467,
│   │   │   │   'end_char': 12791,
│   │   │   │   'parent_text': '\n\n## About `Number`, `String`, `Boolean`, `Symbol` and `Object`\n\nIt can be tempting to think that the types `Number`, `String`, `Boolean`, `Symbol`, or `Object` are the same as the lowercase versions recommended above.\nThese types do not refer to the language primitives however, and almost never should be used as a type.\n\n',
│   │   │   │   'parent_token_count': 82,
│   │   │   │   'parent_start_char': 12467,
│   │   │   │   'parent_end_char': 12791,
│   │   │   │   'parent_level': 2,
│   │   │   │   'grandparent_text': '\n\n## About `Number`, `String`, `Boolean`, `Symbol` and `Object`\n\nIt can be tempting to think that the types `Number`, `String`, `Boolean`, `Symbol`, or `Object` are the same as the lowercase versions recommended above

In [ ]:
context = vs.auto_merge(search,token_budget=128000)

In [ ]:
prompt = COVERAGE_CHECK_PROMPT_TEMPLATE.replace("{sub_queries}", "what does the token separator function call?")
prompt = COVERAGE_CHECK_PROMPT_TEMPLATE.replace("{context}", )

res = llm.astream(input=prompt)

split_prompts = []
async for chunk in res:
    split_prompts.append(chunk.content)
    print(chunk.content, end="", flush=True)